# 14.75 — MLflow Tracking

## Objetivo

Registrar en **MLflow** los experimentos acumulados del proyecto usando como fuente:

- `models/experiments_log.csv`
- `models/final_model.pkl`
- `reports/`

Este notebook permite visualizar en MLflow las métricas, parámetros, artefactos y modelo final del MVP.

## 1. Instalación de dependencias

In [1]:
!pip install mlflow

  Using cached werkzeug-3.1.8-py3-none-any.whl.metadata (4.0 kB)
   ---------------------------------------- 0.0/10.6 MB ? eta -:--:--
   - -------------------------------------- 0.5/10.6 MB 3.6 MB/s eta 0:00:03
   ---- ----------------------------------- 1.3/10.6 MB 3.2 MB/s eta 0:00:03
   ------ --------------------------------- 1.8/10.6 MB 3.0 MB/s eta 0:00:03
   --------- ------------------------------ 2.6/10.6 MB 3.3 MB/s eta 0:00:03
   ------------- -------------------------- 3.7/10.6 MB 3.7 MB/s eta 0:00:02
   ----------------- ---------------------- 4.7/10.6 MB 4.1 MB/s eta 0:00:02
   -------------------- ------------------- 5.5/10.6 MB 3.9 MB/s eta 0:00:02
   --------------------------- ------------ 7.3/10.6 MB 4.5 MB/s eta 0:00:01
   ---------------------------------- ----- 9.2/10.6 MB 5.1 MB/s eta 0:00:01
   ---------------------------------------- 10.6/10.6 MB 5.4 MB/s  0:00:01
   ---------------------------------------- 0.0/3.3 MB ? eta -:--:--
   -------------------------

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress t

## 2. Imports y rutas

In [2]:
import os
import json
import ast
import joblib
import mlflow
import mlflow.sklearn
import pandas as pd

from pathlib import Path

PROJECT_ROOT = Path("..").resolve()

MODELS_DIR = PROJECT_ROOT / "models"
REPORTS_DIR = PROJECT_ROOT / "reports"
MLRUNS_DIR = PROJECT_ROOT / "mlruns"

EXPERIMENTS_LOG_PATH = MODELS_DIR / "experiments_log.csv"
FINAL_MODEL_PATH = MODELS_DIR / "final_model.pkl"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("MODELS_DIR:", MODELS_DIR)
print("REPORTS_DIR:", REPORTS_DIR)
print("MLRUNS_DIR:", MLRUNS_DIR)
print("EXPERIMENTS_LOG_PATH:", EXPERIMENTS_LOG_PATH)
print("FINAL_MODEL_PATH:", FINAL_MODEL_PATH)

PROJECT_ROOT: C:\Users\eduar\dp261-g6
MODELS_DIR: C:\Users\eduar\dp261-g6\models
REPORTS_DIR: C:\Users\eduar\dp261-g6\reports
MLRUNS_DIR: C:\Users\eduar\dp261-g6\mlruns
EXPERIMENTS_LOG_PATH: C:\Users\eduar\dp261-g6\models\experiments_log.csv
FINAL_MODEL_PATH: C:\Users\eduar\dp261-g6\models\final_model.pkl


## 3. Validación de insumos

In [3]:
print("Existe experiments_log.csv:", EXPERIMENTS_LOG_PATH.exists())
print("Existe final_model.pkl:", FINAL_MODEL_PATH.exists())
print("Existe reports/:", REPORTS_DIR.exists())

if not EXPERIMENTS_LOG_PATH.exists():
    raise FileNotFoundError(
        "No existe models/experiments_log.csv. "
        "Primero ejecuta el notebook que genera el log de experimentos."
    )

if not FINAL_MODEL_PATH.exists():
    print("Advertencia: No existe final_model.pkl. Se registrarán métricas, pero no el modelo final.")

Existe experiments_log.csv: True
Existe final_model.pkl: True
Existe reports/: True


## 4. Carga del registro acumulado de experimentos

In [4]:
df_log = pd.read_csv(EXPERIMENTS_LOG_PATH)

print("Filas del log:", len(df_log))
print("Columnas disponibles:")
print(df_log.columns.tolist())

display(df_log.head())

Filas del log: 4
Columnas disponibles:
['timestamp', 'model', 'accuracy', 'precision', 'recall', 'f1', 'roc_auc', 'params', 'model_path', 'selected', 'notes']


,timestamp,model,accuracy,precision,recall,f1,roc_auc,params,model_path,selected,notes
0,2026-05-29 03:15:16,lr,0.7624 ± 0.0027,0.8087 ± 0.0039,0.6875 ± 0.0031,0.7432 ± 0.0029,0.8276 ± 0.0014,baseline_default,../models/baseline_lr.pkl,False,Baseline lineal interpretable.
1,2026-05-29 03:15:16,dt,0.8999 ± 0.0006,0.8947 ± 0.0019,0.9064 ± 0.0014,0.9005 ± 0.0004,0.9009 ± 0.0007,baseline_default,../models/baseline_dt.pkl,False,Baseline de árbol simple.
2,2026-05-29 03:15:16,rf,0.9426 ± 0.0006,0.9494 ± 0.0013,0.9349 ± 0.0009,0.9421 ± 0.0006,0.9826 ± 0.0009,baseline_default,../models/baseline_rf.pkl,True,Modelo seleccionado como mejor candidato para ...
3,2026-05-29 03:15:16,knn,0.8618 ± 0.0020,0.8233 ± 0.0033,0.9214 ± 0.0031,0.8696 ± 0.0017,0.9304 ± 0.0020,baseline_default,../models/baseline_knn.pkl,False,Baseline basado en vecinos cercanos.


## 5. Configuración de MLflow local

In [5]:
mlflow.set_tracking_uri(f"file:///{MLRUNS_DIR.as_posix()}")
mlflow.set_experiment("bank_marketing_mvp")

print("Tracking URI:", mlflow.get_tracking_uri())
print("Experimento activo: bank_marketing_mvp")

C:\Users\eduar\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\mlflow\tracking\_tracking_service\utils.py:184: FutureWarning: The filesystem tracking backend (e.g., './mlruns') is deprecated as of February 2026. Consider transitioning to a database backend (e.g., 'sqlite:///mlflow.db') to take advantage of the latest MLflow features. See https://mlflow.org/docs/latest/self-hosting/migrate-from-file-store for migration guidance.
  return FileStore(store_uri, store_uri)
2026/05/29 03:46:09 INFO mlflow.tracking.fluent: Experiment with name 'bank_marketing_mvp' does not exist. Creating a new experiment.


Tracking URI: file:///C:/Users/eduar/dp261-g6/mlruns
Experimento activo: bank_marketing_mvp


## 6. Funciones auxiliares

In [6]:
def safe_parse_dict(value):
    """Convierte strings tipo diccionario o JSON a dict. Si falla, retorna {}."""
    if pd.isna(value):
        return {}

    if isinstance(value, dict):
        return value

    value = str(value)

    try:
        return json.loads(value)
    except Exception:
        try:
            return ast.literal_eval(value)
        except Exception:
            return {}


def log_metric_if_possible(metric_name, value):
    """Registra una métrica solo si puede convertirse a float."""
    if pd.isna(value):
        return

    try:
        mlflow.log_metric(metric_name, float(value))
    except Exception:
        pass

## 7. Registrar cada fila de `experiments_log.csv` como run

In [7]:
metric_cols = [
    "accuracy",
    "precision",
    "recall",
    "f1",
    "roc_auc",
    "best_score",
    "time_seconds",
    "valor_total_test",
    "valor_por_cliente",
    "impacto_mensual_estimado",
    "impacto_anual_estimado",
]

param_cols = [
    "sprint",
    "stage",
    "experiment_type",
    "scoring",
    "selected",
    "model_path",
]

runs_created = 0

for idx, row in df_log.iterrows():

    model_name = str(row.get("model", f"model_{idx}"))
    sprint = str(row.get("sprint", "unknown_sprint"))
    stage = str(row.get("stage", "unknown_stage"))

    run_name = f"{sprint}_{stage}_{model_name}_{idx}"

    with mlflow.start_run(run_name=run_name):

        mlflow.set_tag("project", "bank_marketing")
        mlflow.set_tag("source", "experiments_log.csv")
        mlflow.set_tag("row_index", idx)

        if "notes" in df_log.columns and pd.notna(row.get("notes")):
            mlflow.set_tag("notes", str(row.get("notes")))

        mlflow.log_param("model", model_name)

        for col in param_cols:
            if col in df_log.columns and pd.notna(row.get(col)):
                mlflow.log_param(col, str(row.get(col)))

        if "params" in df_log.columns:
            params = safe_parse_dict(row.get("params"))
            for key, value in params.items():
                mlflow.log_param(str(key), str(value))

        if "best_params" in df_log.columns:
            best_params = safe_parse_dict(row.get("best_params"))
            for key, value in best_params.items():
                mlflow.log_param(f"best_{key}", str(value))

        for col in metric_cols:
            if col in df_log.columns:
                log_metric_if_possible(col, row.get(col))

        mlflow.log_artifact(str(EXPERIMENTS_LOG_PATH))
        runs_created += 1

print("Runs registrados en MLflow:", runs_created)

Runs registrados en MLflow: 4


## 8. Registrar `final_model.pkl`

In [8]:
if FINAL_MODEL_PATH.exists():

    final_model = joblib.load(FINAL_MODEL_PATH)

    with mlflow.start_run(run_name="final_model_artifact"):

        mlflow.set_tag("project", "bank_marketing")
        mlflow.set_tag("stage", "Sprint 6 MVP")
        mlflow.set_tag("artifact_type", "final_model")

        mlflow.log_param("model_file", "final_model.pkl")
        mlflow.log_param("source", "Sprint 4 final validation + Sprint 5 business value")
        mlflow.log_param("deployment_ready", True)

        mlflow.log_artifact(str(FINAL_MODEL_PATH))

        try:
            mlflow.sklearn.log_model(
                sk_model=final_model,
                artifact_path="model"
            )
            print("Modelo final registrado como MLflow Model.")
        except Exception as e:
            print("Modelo registrado como artifact, pero no como sklearn model.")
            print("Detalle:", e)

else:
    print("No se encontró final_model.pkl. Se omitió el registro del modelo final.")

2026/05/29 03:46:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/29 03:46:12 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/05/29 03:46:13 WARNING mlflow.sklearn: Model was missing function: predict. Not logging python_function flavor!


Modelo final registrado como MLflow Model.


## 9. Registrar artefactos de Business Value

In [9]:
if REPORTS_DIR.exists():

    report_files = [file for file in REPORTS_DIR.glob("*") if file.is_file()]

    with mlflow.start_run(run_name="business_value_artifacts"):

        mlflow.set_tag("project", "bank_marketing")
        mlflow.set_tag("stage", "Sprint 5 Business Value")
        mlflow.log_param("n_report_files", len(report_files))

        for file in report_files:
            mlflow.log_artifact(str(file))

        print("Artefactos de Business Value registrados:", len(report_files))

else:
    print("No existe carpeta reports/.")

Artefactos de Business Value registrados: 7


## 10. Abrir MLflow UI

Después de ejecutar este notebook, abre Git Bash desde el proyecto:

```bash
cd "C:\Users\eduar\dp261-g6"
python -m mlflow ui --backend-store-uri mlruns
```

Luego abre:

```text
http://127.0.0.1:5000
```

En la interfaz se debe visualizar el experimento:

```text
bank_marketing_mvp
```

In [10]:
print("MLflow Tracking completado.")
print("Tracking URI:", mlflow.get_tracking_uri())
print("Carpeta mlruns:", MLRUNS_DIR)
print("Para abrir la UI:")
print("python -m mlflow ui --backend-store-uri mlruns")
print("URL: http://127.0.0.1:5000")

MLflow Tracking completado.
Tracking URI: file:///C:/Users/eduar/dp261-g6/mlruns
Carpeta mlruns: C:\Users\eduar\dp261-g6\mlruns
Para abrir la UI:
python -m mlflow ui --backend-store-uri mlruns
URL: http://127.0.0.1:5000
